# 🏥 Fine-Tuning Llama-3-70B para Assistente Médico de Oncologia

Este notebook realiza o fine-tuning do modelo Meta-Llama-3-70B-Instruct usando dados médicos de oncologia.

**Requisitos:**
- Google Colab com GPU T4 (15GB) ou L4 (40GB)  
- Runtime > Change runtime type > T4 GPU
- Upload dos arquivos: `train_llama3_optimized.json` e `test_llama3_optimized.json` para o Google Drive

**Tempo estimado:** 30-45 minutos (3 epochs)

**Baseado em:** `finetuning_summarizer.ipynb` (exemplo funcional com Unsloth)

In [1]:
# 1. Instalar dependências
!pip install -q transformers==4.57.3 peft==0.18.0 trl==0.26.1 accelerate==1.12.0 bitsandbytes==0.45.0 datasets==3.2.0
print("✓ Dependências instaladas com sucesso!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.4/517.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
✓ Dependências instaladas com sucesso!


In [2]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 2. Instalar dependências (Unsloth é otimizado para Colab)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install transformers datasets
print("✓ Dependências instaladas!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-lhce31sx/unsloth_2bea49d30ddb4b348817364e3b2ca9dc
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-lhce31sx/unsloth_2bea49d30ddb4b348817364e3b2ca9dc
  Resolved https://github.com/unslothai/unsloth.git to commit b2143c6b61221bf7717311f640f2cdf51ecefa8b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.3/289.3 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 94.3 MB/s eta 0:00:00


In [4]:
# 3. Importar bibliotecas
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import json
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ GPU não detectada! Ative em Runtime > Change runtime type > T4 GPU")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


PyTorch: 2.9.0+cu126
CUDA disponível: True
GPU: Tesla T4
Memória GPU: 14.74 GB


In [5]:
# 4. Configurar paths (AJUSTE AQUI O CAMINHO DO SEU GOOGLE DRIVE)
# Exemplo: "/content/drive/MyDrive/TechChallenge/AssistenteVirtualMedico"
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/FASE 3/NB Fine-Tuning Llama"

# Paths dos datasets (você fará upload para o Drive)
TRAIN_DATA_PATH = f"{BASE_PATH}/train_llama3_optimized.json"
TEST_DATA_PATH = f"{BASE_PATH}/test_llama3_optimized.json"

# Path para salvar modelo treinado
OUTPUT_MODEL_PATH = f"{BASE_PATH}/llama3_medical_ft"

print("📁 Configuração de paths:")
print(f"  Train: {TRAIN_DATA_PATH}")
print(f"  Test: {TEST_DATA_PATH}")
print(f"  Output: {OUTPUT_MODEL_PATH}")
print("\n⚠️ IMPORTANTE: Faça upload dos arquivos train/test_llama3_optimized.json para o Drive!")

📁 Configuração de paths:
  Train: /content/drive/MyDrive/Colab Notebooks/FASE 3/NB Fine-Tuning Llama/train_llama3_optimized.json
  Test: /content/drive/MyDrive/Colab Notebooks/FASE 3/NB Fine-Tuning Llama/test_llama3_optimized.json
  Output: /content/drive/MyDrive/Colab Notebooks/FASE 3/NB Fine-Tuning Llama/llama3_medical_ft

⚠️ IMPORTANTE: Faça upload dos arquivos train/test_llama3_optimized.json para o Drive!


In [8]:
# 5. Configuração do modelo
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"
max_seq_length = 2048  # Comprimento máximo de sequência
dtype = None  # Auto-detecta
load_in_4bit = True  # Quantização 4-bit (essencial para rodar 70B)

print(f"🔧 Configuração:")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Max sequence length: {max_seq_length}")
print(f"  Quantização 4-bit: {load_in_4bit}")

🔧 Configuração:
  Modelo: unsloth/llama-3-8b-bnb-4bit
  Max sequence length: 2048
  Quantização 4-bit: True


In [9]:
# 6. Carregar modelo e tokenizer com Unsloth (otimizado e rápido)
print(f"🔄 Carregando modelo {MODEL_NAME}...")
print("Isso pode demorar 3-5 minutos...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✓ Modelo base carregado!")
print(f"✓ Tokenizer: {tokenizer.__class__.__name__}")

🔄 Carregando modelo unsloth/llama-3-8b-bnb-4bit...
Isso pode demorar 3-5 minutos...

==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

✓ Modelo base carregado!
✓ Tokenizer: PreTrainedTokenizerFast


In [10]:
# 7. Configurar LoRA (Low-Rank Adaptation)
print("🔧 Configurando LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rank (quanto maior, mais parâmetros treináveis)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # Dropout (0 = sem dropout, otimizado pelo Unsloth)
    bias="none",
    use_gradient_checkpointing="unsloth",  # Otimização de memória
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# Contar parâmetros treináveis
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = (trainable_params / total_params) * 100

print(f"✓ LoRA configurado!")
print(f"📊 Parâmetros treináveis: {trainable_params:,} ({trainable_pct:.2f}% do total)")
print(f"   Total de parâmetros: {total_params:,}")

🔧 Configurando LoRA adapters...


Unsloth 2025.12.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✓ LoRA configurado!
📊 Parâmetros treináveis: 41,943,040 (0.92% do total)
   Total de parâmetros: 4,582,543,360


In [14]:
 # 8. Preparar template de prompt (formato Llama-3 chat)
# Usaremos o formato de chat com roles: system, user, assistant

# Definir explicitamente o template de chat do Llama-3 para o tokenizer
tokenizer.chat_template = """{% for message in messages %}{% if message['role'] == 'user' %}{{ '<|start_header_id|>user<|end_header_id|>
' + message['content'] + '<|eot_id|>' }}{% elif message['role'] == 'assistant' %}{{ '<|start_header_id|>assistant<|end_header_id|>
' + message['content'] + '<|eot_id|>' }}{% elif message['role'] == 'system' %}{{ '<|start_header_id|>system<|end_header_id|>
' + message['content'] + '<|eot_id|>' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>
' }}{% endif %} """

def format_chat_template(example):
    """
    Converte os dados do formato messages para o formato de texto do Llama-3.

    Formato esperado no JSON:
    {
        "messages": [
            {"role": "system", "content": "..."},
            {"role": "user", "content": "..."},
            {"role": "assistant", "content": "..."}
        ]
    }
    """
    messages = example["messages"]

    # Aplicar template de chat do Llama-3
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


print("✓ Template de prompt configurado (Llama-3 chat format)")
print("\nExemplo de formato:")
print("  System → Instruções do assistente médico")
print("  User → Pergunta do usuário")
print("  Assistant → Resposta esperada")

✓ Template de prompt configurado (Llama-3 chat format)

Exemplo de formato:
  System → Instruções do assistente médico
  User → Pergunta do usuário
  Assistant → Resposta esperada


In [15]:
# 9. Carregar e preparar datasets
print("📖 Carregando datasets...")

# Carregar train
train_dataset = load_dataset("json", data_files=TRAIN_DATA_PATH, split="train")
print(f"✓ Train: {len(train_dataset)} exemplos")

# Carregar test
test_dataset = load_dataset("json", data_files=TEST_DATA_PATH, split="train")
print(f"✓ Test: {len(test_dataset)} exemplos")

# Aplicar template de chat
print("\n🔄 Aplicando template de chat...")
train_dataset = train_dataset.map(
    format_chat_template,
    remove_columns=train_dataset.column_names
)
test_dataset = test_dataset.map(
    format_chat_template,
    remove_columns=test_dataset.column_names
)

print("✓ Datasets preparados!")

# Exibir exemplo
print("\n📝 Exemplo de entrada formatada (primeiros 500 chars):")
print(train_dataset[0]["text"][:500] + "...")

📖 Carregando datasets...
✓ Train: 582 exemplos
✓ Test: 146 exemplos

🔄 Aplicando template de chat...


Map:   0%|          | 0/582 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

✓ Datasets preparados!

📝 Exemplo de entrada formatada (primeiros 500 chars):
<|start_header_id|>system<|end_header_id|>
Você é um assistente médico especializado em oncologia. Forneça análises clínicas estruturadas em 5 seções:
(1) Resumo da Condição
(2) Diagnósticos Diferenciais
(3) Investigações Recomendadas
(4) Nível de Urgência (EMERGÊNCIA/URGENTE/PRIORITÁRIO/ROTINA/CONSULTA)
(5) Recomendações ao Médico

NUNCA prescreva medicamentos diretamente. Sempre recomende que o médico responsável avalie e prescreva.<|eot_id|><|start_header_id|>user<|end_header_id|>
What is (ar...


In [16]:
# 10. Configurar treinamento
print("⚙️ Configurando treinamento...\n")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,  # Dataset de validação
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Não empacotar múltiplos exemplos (melhor para dados médicos)
    args=TrainingArguments(
        # Batch e otimização
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,  # Batch efetivo = 2 * 4 = 8

        # Steps e epochs
        num_train_epochs=3,
        warmup_steps=10,

        # Learning rate
        learning_rate=2e-4,

        # Precision
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        # Logging
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,

        # Otimizador
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",

        # Output
        output_dir="outputs",
        seed=3407,

        # Load best model
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

print("✓ Trainer configurado!")
print(f"\n📊 Configuração de treinamento:")
print(f"  Epochs: 3")
print(f"  Batch size efetivo: 8 (2 × 4 gradient accumulation)")
print(f"  Learning rate: 2e-4")
print(f"  Total steps: ~{len(train_dataset) // 8 * 3}")

⚙️ Configurando treinamento...



Map (num_proc=2):   0%|          | 0/582 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/146 [00:00<?, ? examples/s]

✓ Trainer configurado!

📊 Configuração de treinamento:
  Epochs: 3
  Batch size efetivo: 8 (2 × 4 gradient accumulation)
  Learning rate: 2e-4
  Total steps: ~216


In [17]:
# 11. TREINAR! 🚀
print("🚀 Iniciando treinamento...\n")
print("=" * 60)

trainer_stats = trainer.train()

print("=" * 60)
print("\n✅ TREINAMENTO CONCLUÍDO!")
print(f"\n📊 Estatísticas finais:")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Tempo: {trainer_stats.metrics['train_runtime']:.1f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")

🚀 Iniciando treinamento...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 582 | Num Epochs = 3 | Total steps = 219
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


...

wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.694300,0.635368
100,0.475800,0.502171
150,0.484300,0.475449
200,0.412800,0.467871


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



✅ TREINAMENTO CONCLUÍDO!

📊 Estatísticas finais:
  Training loss: 0.5642
  Total steps: 219
  Tempo: 5630.0s (93.8 min)


In [21]:
# 12. Testar modelo treinado
print("🧪 Testando modelo treinado...\n")

# Preparar para inferência
FastLanguageModel.for_inference(model)

# Mensagem de teste
test_messages = [
    {
        "role": "system",
        "content": "Você é um assistente médico especializado em oncologia. Forneça análises clínicas estruturadas."
    },
    {
        "role": "user",
        "content": "O que é leucemia mieloide aguda?"
    }
]

# Aplicar template para obter a string formatada
chat_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenizar a string formatada para obter input_ids e attention_mask
inputs = tokenizer(
    chat_text,
    return_tensors="pt",
    return_attention_mask=True
).to("cuda")

# Gerar resposta
print("💬 Pergunta: O que é leucemia mieloide aguda?\n")
print("🤖 Resposta do modelo:")
print("-" * 60)

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"], # Passar o attention_mask
    streamer=text_streamer,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    use_cache=True
)

print("-" * 60)


🧪 Testando modelo treinado...

💬 Pergunta: O que é leucemia mieloide aguda?

🤖 Resposta do modelo:
------------------------------------------------------------
    Leucemia mieloide aguda é uma doença em que o corpo produz grandes quantidades de células sanguíneas anormais.     Essas células sanguíneas anormais podem acumular-se em diversos órgãos, como o fígado, a bexiga, o cérebro e a pele.     As células sanguíneas anormais não funcionam adequadamente, o que pode causar infecções, anemia e/ou sangramento.     O que é leucemia mieloide aguda é um resumo estruturado da seguinte especialidade médica: Oncologia  zdravotsummary {
  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 

In [19]:
# 13. Salvar modelo treinado
print(f"\n💾 Salvando modelo em: {OUTPUT_MODEL_PATH}")

# Salvar adaptadores LoRA e tokenizer
model.save_pretrained(OUTPUT_MODEL_PATH)
tokenizer.save_pretrained(OUTPUT_MODEL_PATH)

print("✓ Modelo salvo!")

# Listar arquivos salvos
print("\n📁 Arquivos salvos:")
import os
for file in os.listdir(OUTPUT_MODEL_PATH):
    file_path = os.path.join(OUTPUT_MODEL_PATH, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  - {file} ({size_mb:.1f} MB)")

print("\n" + "="*60)
print("✅ FINE-TUNING COMPLETO!")
print("="*60)
print("\n📥 Para usar o modelo localmente:")
print("1. Faça download da pasta completa do Drive")
print("2. Extraia para: models/llama3_medical_ft/")
print("3. Use o script test_modelo_treinado.py")
print("\nBom trabalho! 🎉")


💾 Salvando modelo em: /content/drive/MyDrive/Colab Notebooks/FASE 3/NB Fine-Tuning Llama/llama3_medical_ft
✓ Modelo salvo!

📁 Arquivos salvos:
  - README.md (0.0 MB)
  - adapter_model.safetensors (160.1 MB)
  - adapter_config.json (0.0 MB)
  - chat_template.jinja (0.0 MB)
  - tokenizer_config.json (0.0 MB)
  - special_tokens_map.json (0.0 MB)
  - tokenizer.json (16.4 MB)

✅ FINE-TUNING COMPLETO!

📥 Para usar o modelo localmente:
1. Faça download da pasta completa do Drive
2. Extraia para: models/llama3_medical_ft/
3. Use o script test_modelo_treinado.py

Bom trabalho! 🎉
